In [ ]:
from typing import Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from kagraph import END, START, MessagesState, StateGraph, invoke_llm
from kagraph.llms import load_llm
from kagraph.messages import AIMessage, HumanMessage
from kagraph.tracing import trace

In [ ]:
class GenerateDecision(BaseModel):
    essay: str = Field(description="The generated or revised essay.")
    next_step: Literal["reflect", "end"] = Field(
        description=(
            "Choose 'reflect' if the essay should receive one more critique. "
            "Choose 'end' if the latest essay is ready to accept."
        )
    )
    reason: str = Field(description="Brief reason for the routing decision.")


class ReflectionState(MessagesState, total=False):
    next_step: Literal["reflect", "end"]
    route_reason: str

class ReflectionInput(TypedDict):
    input: str

In [ ]:
GENERATION_SYSTEM = (
    "You are an essay assistant tasked with writing clear, compact essays. "
    "Generate the best essay possible for the user's request. If the user "
    "provides critique, revise the previous attempt accordingly. After writing, "
    "decide whether another critique/revision cycle is useful."
)

REFLECTION_SYSTEM = (
    "You are a teacher grading an essay submission. Generate specific critique "
    "and revision recommendations. Be direct and actionable."
)

In [ ]:
def build_graph(llm):
    graph = StateGraph(ReflectionState, input_schema=ReflectionInput)

    def prepare_input(state: ReflectionInput):
        return {"messages": [HumanMessage(state["input"])]}

    def generation_node(state: ReflectionState):
        decision_message = invoke_llm(
            llm,
            messages=state["messages"],
            prompt=(
                "Write or revise the essay now. Keep it under 450 words.\n\n"
                "Then decide the next graph step:\n"
                "- next_step='reflect' if another teacher critique would materially improve it.\n"
                "- next_step='end' if the latest essay is ready to accept.\n"
                "Prefer reflection for an unrevised first draft."
            ),
            system=GENERATION_SYSTEM,
            schema=GenerateDecision,
        )
        decision = decision_message.content

        return {
            "messages": [AIMessage(decision.essay)],
            "next_step": decision.next_step,
            "route_reason": decision.reason,
        }

    def reflection_node(state: ReflectionState):
        critique = invoke_llm(
            llm,
            messages=state["messages"],
            prompt="Review the latest essay in this conversation and provide critique.",
            system=REFLECTION_SYSTEM,
        ).text
        return {"messages": [HumanMessage(critique)]}

    def should_continue(state: ReflectionState):
        return END if state.get("next_step") == "end" else "reflect"

    graph.add_node("prepare_input", prepare_input, input_schema=ReflectionInput)
    graph.add_node("generate", generation_node)
    graph.add_node("reflect", reflection_node)
    graph.add_edge(START, "prepare_input")
    graph.add_edge("prepare_input", "generate")
    # Explicit mapping prevents the visualizer from drawing fallback edges to all nodes
    graph.add_conditional_edges("generate", should_continue, {END: END, "reflect": "reflect"})
    graph.add_edge("reflect", "generate")
    return graph.compile(
        name="reflection",
    )

In [ ]:
llm = load_llm("openai/gpt-5.4-mini-2026-03-17", support_structured_outputs=True)
app = build_graph(llm)
app.get_graph().draw_png()

In [ ]:
with trace("Reflection"):
    result = app.invoke(
        {"input": "Write an short and concise essay on why The Little Prince remains relevant in modern childhood."},
        chat_name="KaGraph Reflection tutorial",
        recursion_limit=10,
    )

In [ ]:
result['chat']